# state_conditioned_multi_objective_ytd_gate_v1

Preflight card for the next trainable candidate from `retention_ytd_training_objective_handoff_v1`. The candidate is not promoted and may not retain an actor until official `wf_2020`, `wf_2022`, `wf_2023_2024`, and YTD gates pass.

In [1]:
from __future__ import annotations

import json
import sys
from datetime import datetime, timezone
from pathlib import Path

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists() and (path / "src").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.config import load_profile
from src.soxl_quant.data import feature_columns_for_packs, feature_schema_signature
from src.soxl_quant.registry import load_profile_registry, validate_profile_registry

EXPERIMENT = "state_conditioned_multi_objective_ytd_gate_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
HANDOFF = ROOT / "runtime" / "model_selection" / "card" / "retention_ytd_training_objective_handoff_v1" / "selection.json"
RETENTION_GAP = ROOT / "runtime" / "model_selection" / "card" / "retention_gap_aggregate_v1" / "selection.json"

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))


In [2]:
handoff = read_json(HANDOFF)
retention_gap = read_json(RETENTION_GAP)
profile = load_profile(EXPERIMENT)
registry_entry = load_profile_registry()[EXPERIMENT]
registry_issues = validate_profile_registry()
feature_columns = feature_columns_for_packs(profile.feature.packs, profile.name)

preflight = {
    "experiment": EXPERIMENT,
    "profile_name": profile.name,
    "registry_status": registry_entry.status,
    "registry_baseline": registry_entry.baseline,
    "registry_keep_actor": registry_entry.keep_actor,
    "registry_keep_summary": registry_entry.keep_summary,
    "registry_notes_ref": registry_entry.notes_ref,
    "registry_issues": registry_issues,
    "feature_packs": list(profile.feature.packs),
    "feature_count": len(feature_columns),
    "feature_schema_signature": feature_schema_signature(profile.feature.packs, profile.name),
    "changed_layer": ["feature", "reward", "selection"],
    "canonical_baseline": handoff["canonical_baseline"],
    "direct_comparison_baseline": registry_entry.baseline,
    "reference_baseline": handoff["reference_baseline"],
    "target_window": handoff["target_thresholds"]["required_wf_windows"],
    "hard_wf_max_drawdown_pct": profile.promotion_gate.max_test_2022_drawdown,
    "min_wf_relative_outperformance_pp": profile.promotion_gate.min_wf_relative_outperformance,
    "ytd_current_best_alignment_pct": handoff["target_thresholds"]["required_current_best_alignment_ytd_pct"],
    "soxlbacktest_reference_ytd_return_pct": handoff["target_thresholds"]["soxlbacktest_reference_ytd_return_pct"],
    "safe_backtest_end": handoff["target_thresholds"]["safe_backtest_end"],
    "closest_success_case": retention_gap["best_wf_drawdown_pass_candidate_by_ytd"]["candidate"],
    "closest_success_gap": retention_gap["best_wf_drawdown_pass_candidate_by_ytd"]["failures"],
    "closest_failure_case": retention_gap["best_complete_candidate"]["candidate"],
    "closest_failure_gap": retention_gap["best_complete_candidate"]["failures"],
    "hypothesis": "State-conditioned reward plus trend-preservation and drawdown-drift terms can keep the best complete candidate's participation while reducing WF drawdown below 20pct.",
    "acceptance_criteria": [
        "official wf_2020/wf_2022/wf_2023_2024 metrics are present",
        "worst official WF MaxDD < 20pct",
        "YTD return >= SOXLBacktest buy/sell reference and >= 95pct of current best",
        "checkpoint selector table includes retention_ytd_loss and wf_drawdown_loss",
        "drawdown interval audit exists for selected checkpoint",
    ],
    "expected_tradeoff": "More conservative stress-state positioning may reduce peak YTD return; trend-preservation terms are intended to prevent the low-drawdown underparticipation failure mode.",
    "failure_mode": "Either WF drawdown stays above 20pct like mode_conditioned_rebound_reward_train_v2, or the new drawdown terms collapse YTD participation like wf_pass_ytd18_15_15_02.",
}

preflight


{'experiment': 'state_conditioned_multi_objective_ytd_gate_v1',
 'profile_name': 'state_conditioned_multi_objective_ytd_gate_v1',
 'registry_status': 'candidate',
 'registry_baseline': 'mode_conditioned_rebound_reward_train_v2',
 'registry_keep_actor': False,
 'registry_keep_summary': True,
 'registry_notes_ref': 'runtime/model_selection/card/state_conditioned_multi_objective_ytd_gate_v1/state_conditioned_multi_objective_ytd_gate_v1.ipynb',
 'registry_issues': [],
 'feature_packs': ['vix_term',
  'transition_signals',
  'soxx_spread_oscillation'],
 'feature_count': 79,
 'feature_schema_signature': '2f64801809a1',
 'changed_layer': ['feature', 'reward', 'selection'],
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'mode_conditioned_rebound_reward_train_v2',
 'reference_baseline': 'soxlbacktest_policy_reimplementation_v1',
 'target_window': ['wf_2020', 'wf_2022', 'wf_2023_2024'],
 'hard_wf_max_drawdown_pct': 20.0,
 'min_wf_relative_outperfor

In [3]:
assert preflight["registry_status"] == "candidate"
assert preflight["registry_keep_actor"] is False
assert preflight["direct_comparison_baseline"] == "mode_conditioned_rebound_reward_train_v2"
assert preflight["hard_wf_max_drawdown_pct"] <= 20.0
assert preflight["min_wf_relative_outperformance_pp"] >= 30.0
assert profile.reward.state_conditioned_reward_enabled is True
assert profile.reward.drawdown_drift_reward_enabled is True
assert profile.reward.trend_preservation_floor_enabled is True
assert profile.selection_score.use_pareto is True
assert profile.checkpoint_retention.enabled is True
assert registry_issues == []

created_at = datetime.now(timezone.utc).isoformat()
summary = {
    "experiment": EXPERIMENT,
    "created_at": created_at,
    "decision": "candidate_profile_ready_for_smoke_training_no_promotion",
    "promotion_allowed": False,
    "actor_retention_allowed": False,
    "training_allowed_next": True,
    "profile_yaml": f"runtime/model_selection/card/{EXPERIMENT}/profile.yaml",
    "profile_preflight": f"runtime/model_selection/card/{EXPERIMENT}/profile_preflight.json",
    "source_handoff": str(HANDOFF.relative_to(ROOT)),
    "source_retention_gap": str(RETENTION_GAP.relative_to(ROOT)),
    "target_thresholds": {
        "required_wf_windows": preflight["target_window"],
        "wf_max_drawdown_hard_gate_pct": preflight["hard_wf_max_drawdown_pct"],
        "official_internal_relative_return_floor_pp": preflight["min_wf_relative_outperformance_pp"],
        "soxlbacktest_reference_ytd_return_pct": preflight["soxlbacktest_reference_ytd_return_pct"],
        "required_current_best_alignment_ytd_pct": preflight["ytd_current_best_alignment_pct"],
        "safe_backtest_end": preflight["safe_backtest_end"],
    },
    "candidate_profile": {
        "name": profile.name,
        "baseline": registry_entry.baseline,
        "feature_packs": preflight["feature_packs"],
        "feature_count": preflight["feature_count"],
        "feature_schema_signature": preflight["feature_schema_signature"],
        "changed_layer": preflight["changed_layer"],
    },
    "next_required_artifacts": handoff["required_next_artifacts"],
}
selection = dict(summary)
selection.update({
    "closest_success_case": preflight["closest_success_case"],
    "closest_failure_case": preflight["closest_failure_case"],
    "hypothesis": preflight["hypothesis"],
    "acceptance_criteria": preflight["acceptance_criteria"],
    "expected_tradeoff": preflight["expected_tradeoff"],
    "failure_mode": preflight["failure_mode"],
})

(CARD / "profile_preflight.json").write_text(json.dumps(preflight, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(CARD / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True) + "\n", encoding="utf-8")

summary


{'experiment': 'state_conditioned_multi_objective_ytd_gate_v1',
 'created_at': '2026-06-13T15:47:51.169531+00:00',
 'decision': 'candidate_profile_ready_for_smoke_training_no_promotion',
 'promotion_allowed': False,
 'actor_retention_allowed': False,
 'training_allowed_next': True,
 'profile_yaml': 'runtime/model_selection/card/state_conditioned_multi_objective_ytd_gate_v1/profile.yaml',
 'profile_preflight': 'runtime/model_selection/card/state_conditioned_multi_objective_ytd_gate_v1/profile_preflight.json',
 'source_handoff': 'runtime/model_selection/card/retention_ytd_training_objective_handoff_v1/selection.json',
 'source_retention_gap': 'runtime/model_selection/card/retention_gap_aggregate_v1/selection.json',
 'target_thresholds': {'required_wf_windows': ['wf_2020',
   'wf_2022',
   'wf_2023_2024'],
  'wf_max_drawdown_hard_gate_pct': 20.0,
  'official_internal_relative_return_floor_pp': 30.0,
  'soxlbacktest_reference_ytd_return_pct': 132.077978,
  'required_current_best_alignment_

## Conclusion

Decision: `candidate_profile_ready_for_smoke_training_no_promotion`.

This card creates the next trainable profile requested by the retention handoff. It is deliberately registered with `keep_actor: false` so dashboard/model replay discovery cannot surface it as usable before official evidence exists. The candidate differs from the closest failed reward diagnostics by adding an explicit feature schema (`vix_term`, `transition_signals`, `soxx_spread_oscillation`), Pareto checkpoint selection, a 20% max-drawdown gate, drawdown-drift penalty, and trend-preservation floor.

Next action: run smoke training first, then official `wf_2020`, `wf_2022`, `wf_2023_2024`, dashboard/YTD replay, checkpoint selector table with `retention_ytd_loss` and `wf_drawdown_loss`, and selected-checkpoint drawdown interval audit. Promotion and actor retention remain blocked until those artifacts pass.

## 2026-06-13 Training Run Evidence

This continuation ran the candidate through smoke and non-smoke training after fixing replay export to use the profile-specific 79-column feature schema. The profile remains non-promotable and no actor may be retained.

In [4]:
import json
from datetime import datetime, timezone
from pathlib import Path

card = Path.cwd()
summary_path = card / "summary.json"
if not summary_path.exists():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "runtime/model_selection/card/state_conditioned_multi_objective_ytd_gate_v1/summary.json"
        if candidate.exists():
            summary_path = candidate
            card = candidate.parent
            break
summary = json.loads(summary_path.read_text())

training_evidence = {
    "updated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "decision": "full_training_low_drawdown_underparticipation_no_actor_retention",
    "export_replay_fix": {
        "script": "scripts/export_profile_daily_backtest.py",
        "behavior": "build_dataset(feature_packs=profile.feature.packs, profile_name=profile.name) and pass feature_columns/mode_router_config to backtest_profile",
        "regression_test": "tests/test_artifact_paths.py::test_export_profile_daily_backtest_uses_profile_specific_feature_packs",
    },
    "smoke_training": {
        "exit_code": 0,
        "feature_count": 79,
        "valid_return_pct": 19.249673389962794,
        "valid_sharpe": 1.0442795056470329,
        "valid_max_dd_pct": 16.1256678708669,
        "test_proxy_return_pct": 445.9026548861792,
        "test_proxy_sharpe": 1.0398150049554507,
        "test_proxy_max_dd_pct": 37.48769144071225,
        "test_2022_max_dd_pct": 35.16293192654828,
        "promotion": "blocked_fixed_split_drawdown",
        "explicit_replay_ytd_return_pct": 67.20440245729888,
        "explicit_replay_ytd_max_dd_pct": 13.21597504652031,
    },
    "full_training": {
        "exit_code": 0,
        "feature_count": 79,
        "episodes": 154,
        "selection_winner_iter": 0,
        "valid_return_pct": 5.7928704148944465,
        "valid_sharpe": 0.5144474514827932,
        "valid_max_dd_pct": 13.309654079740898,
        "test_proxy_return_pct": 188.13510663063994,
        "test_proxy_sharpe": 1.0679053039575948,
        "test_proxy_max_dd_pct": 16.563316750310896,
        "test_proxy_relative_return_pp": -2387.368414916369,
        "test_2020": {"return_pct": 26.040639622431417, "max_dd_pct": 16.22290216880506, "relative_return_pp": -329.69552010272724},
        "test_2022": {"return_pct": -11.877414582653056, "max_dd_pct": 14.078273680027062, "relative_return_pp": 73.31970192520201},
        "test_2023_2024": {"return_pct": 23.48713830351126, "max_dd_pct": 10.904924929821144, "relative_return_pp": -32.16733540518679},
        "fixed_split_gate": "passed",
        "promotion": "blocked_missing_walk_forward_and_ytd_underparticipation",
    },
    "explicit_full_selected_replay": {
        "rows": 1619,
        "start": "2020-01-02",
        "end": "2026-06-11",
        "return_pct": 126.14093558897008,
        "buyhold_return_pct": 1050.7911909972836,
        "relative_return_pp": -924.6502554083136,
        "max_dd_pct": 18.528988813487622,
        "avg_pos": 0.121072870302098,
        "trades": 160,
    },
    "explicit_ytd_2026_replay": {
        "rows": 111,
        "start": "2026-01-02",
        "end": "2026-06-11",
        "return_pct": 18.966005478709192,
        "buyhold_return_pct": 374.15325473043845,
        "relative_return_pp": -355.18724925172927,
        "max_dd_pct": 6.573022638581703,
        "avg_pos": 0.04855675811762275,
        "trades": 14,
    },
    "target_thresholds": {
        "wf_max_drawdown_hard_gate_pct": 20.0,
        "soxlbacktest_reference_ytd_return_pct": 132.077978,
        "required_current_best_alignment_ytd_pct": 213.143462,
    },
    "gate_result": {
        "fixed_split_gate": True,
        "full_replay_maxdd_gate": True,
        "ytd_maxdd_gate": True,
        "ytd_soxlbacktest_return_gate": False,
        "ytd_current_best_alignment_gate": False,
        "walk_forward_gate": False,
    },
    "actor_retention_allowed": False,
    "promotion_allowed": False,
    "cleanup": "generated actor/checkpoint/daily CSV artifacts are not retained; rerun notebook commands to reproduce",
    "next_mechanism": "revise trend/YTD participation objective before another PPO branch; this profile solved drawdown by staying too flat",
}

summary["decision"] = training_evidence["decision"]
summary["training_allowed_next"] = False
summary["promotion_allowed"] = False
summary["actor_retention_allowed"] = False
summary["latest_training_evidence"] = training_evidence
summary["next_required_artifacts"] = [
    "new or revised objective that lifts YTD participation without losing <20pct drawdown evidence",
    "official wf_2020/wf_2022/wf_2023_2024 metrics for any successor candidate",
    "dashboard/YTD replay through safe_backtest_end for any successor candidate",
]
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n")

{
    "decision": training_evidence["decision"],
    "full_test_proxy": training_evidence["full_training"],
    "explicit_ytd_2026_replay": training_evidence["explicit_ytd_2026_replay"],
    "gate_result": training_evidence["gate_result"],
    "next_mechanism": training_evidence["next_mechanism"],
}


{'decision': 'full_training_low_drawdown_underparticipation_no_actor_retention',
 'full_test_proxy': {'exit_code': 0,
  'feature_count': 79,
  'episodes': 154,
  'selection_winner_iter': 0,
  'valid_return_pct': 5.7928704148944465,
  'valid_sharpe': 0.5144474514827932,
  'valid_max_dd_pct': 13.309654079740898,
  'test_proxy_return_pct': 188.13510663063994,
  'test_proxy_sharpe': 1.0679053039575948,
  'test_proxy_max_dd_pct': 16.563316750310896,
  'test_proxy_relative_return_pp': -2387.368414916369,
  'test_2020': {'return_pct': 26.040639622431417,
   'max_dd_pct': 16.22290216880506,
   'relative_return_pp': -329.69552010272724},
  'test_2022': {'return_pct': -11.877414582653056,
   'max_dd_pct': 14.078273680027062,
   'relative_return_pp': 73.31970192520201},
  'test_2023_2024': {'return_pct': 23.48713830351126,
   'max_dd_pct': 10.904924929821144,
   'relative_return_pp': -32.16733540518679},
  'fixed_split_gate': 'passed',
  'promotion': 'blocked_missing_walk_forward_and_ytd_underpar

### Training Conclusion

Decision: `full_training_low_drawdown_underparticipation_no_actor_retention`.

The non-smoke run improved the drawdown side enough for fixed-split checks: test proxy return `188.14%`, Sharpe `1.068`, MaxDD `16.56%`; `test_2022` MaxDD was `14.08%`. The explicit replay also kept full-window MaxDD under 20% (`18.53%`) and YTD MaxDD low (`6.57%`).

The candidate fails the actual retention objective because it stayed too flat: explicit 2026 YTD return was only `18.97%`, below both the SOXLBacktest reference floor `132.08%` and the current-best alignment floor `213.14%`. It also has no official walk-forward evidence yet. Actor retention and promotion remain blocked; the generated actor/checkpoint files are disposable smoke/full-run artifacts.

Next: revise the trend/YTD participation objective before another PPO branch. This profile is useful as branch-closure evidence: the reward stack can hit drawdown by underparticipating, but it does not solve the requested return alignment.